# Trend Model Keywords

Untuk setiap **model** dan **subjek**, ambil topik-topik yang dikategorikan sebagai
**GROWING**, **STABLE**, dan **DECLINING** dari `topic_trends.csv`, lalu kumpulkan
semua kata dari topik-topik tersebut via `topic_word_evolution.csv`.

**Output per (model, subjek, kategori):**
- Top-20 kata dengan frekuensi kemunculan tertinggi di seluruh (tahun × topik) dalam kategori tersebut

**Output file:**
- `results/shared/trend_model/{model}/{subject}/trend_top_words.csv`
- `results/shared/trend_model/trend_top_words_all.csv` — gabungan semua model × subjek
- `results/shared/trend_model/trend_top_words_subject.csv` — agregasi lintas model per subjek

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

## Configuration

In [2]:
BASE = Path("/home/nedo/Kuliah/TA/Program")
RESULTS_DIR = BASE / "results"

LIST_SUBJECT = ["cs", "math", "physics"]
MODELS = ["dtm", "lda", "top2vec", "bertopic", "topicGpt"]
MODEL_LABELS = {
    "dtm": "DTM", "lda": "LDA", "top2vec": "Top2Vec",
    "bertopic": "BERTopic", "topicGpt": "TopicGPT",
}
TREND_CATEGORIES = ["GROWING", "STABLE", "DECLINING"]
TOP_N = 20  # top words per category

# Create output directories
for model in MODELS:
    for subject in LIST_SUBJECT:
        (RESULTS_DIR / "shared" / "trend_model" / model / subject).mkdir(parents=True, exist_ok=True)

print(f"Models : {list(MODEL_LABELS.values())}")
print(f"Subjects: {LIST_SUBJECT}")
print(f"Top-N words per category: {TOP_N}")

Models : ['DTM', 'LDA', 'Top2Vec', 'BERTopic', 'TopicGPT']
Subjects: ['cs', 'math', 'physics']
Top-N words per category: 20


## Helper Functions

In [3]:
def load_trend_topics(model: str, subject: str) -> dict:
    """
    Load topic_trends.csv dan kembalikan dict:
        {trend_category: set_of_topic_ids}
    trend_category: 'GROWING' | 'STABLE' | 'DECLINING'
    """
    path = RESULTS_DIR / model / "temporal" / subject / "topic_trends.csv"
    if not path.exists():
        return {cat: set() for cat in TREND_CATEGORIES}

    df = pd.read_csv(path)
    df["trend"] = df["trend"].str.upper().str.strip()

    result = {}
    for cat in TREND_CATEGORIES:
        result[cat] = set(df[df["trend"] == cat]["topic_id"].tolist())
    return result


def load_word_evolution(model: str, subject: str) -> pd.DataFrame:
    """
    Load topic_word_evolution.csv.
    Kolom: subject, year, topic_id, top_words
    """
    path = RESULTS_DIR / model / "temporal" / subject / "topic_word_evolution.csv"
    if not path.exists():
        return pd.DataFrame(columns=["year", "topic_id", "top_words"])
    df = pd.read_csv(path)
    return df


def top_words_for_category(
    trend_topic_ids: set,
    evo_df: pd.DataFrame,
    top_n: int = 20,
) -> list:
    """
    Dari semua baris evolution yang topic_id-nya ada di trend_topic_ids,
    kumpulkan semua kata (per kemunculan di tiap tahun x topik),
    lalu kembalikan top_n kata berdasarkan frekuensi terbanyak.

    Returns: [(word, freq), ...] sorted by freq desc
    """
    if not trend_topic_ids or evo_df.empty:
        return []

    subset = evo_df[evo_df["topic_id"].isin(trend_topic_ids)]
    counter = Counter()

    for top_words_str in subset["top_words"]:
        words = [w.strip() for w in str(top_words_str).split(",") if w.strip()]
        counter.update(words)

    return counter.most_common(top_n)

## Main: Compute Top-20 Words per Model × Subject × Trend Category

In [4]:
ICON = {"GROWING": "📈", "STABLE": "🔒", "DECLINING": "📉"}

# Kumpulkan semua hasil untuk disimpan ke satu CSV gabungan
all_rows = []

for model in MODELS:
    label = MODEL_LABELS[model]
    print(f"\n{'='*70}")
    print(f"Model: {label}")
    print(f"{'='*70}")

    for subject in LIST_SUBJECT:
        print(f"\n  [{subject.upper()}]")

        # Load data
        trend_topics = load_trend_topics(model, subject)
        evo_df = load_word_evolution(model, subject)

        if evo_df.empty:
            print(f"    ⚠ No evolution data found — skipping")
            continue

        # Summary jumlah topik per kategori
        for cat in TREND_CATEGORIES:
            n = len(trend_topics[cat])
            print(f"    {ICON[cat]} {cat}: {n} topics")

        per_subject_rows = []

        for cat in TREND_CATEGORIES:
            top_words = top_words_for_category(trend_topics[cat], evo_df, top_n=TOP_N)

            if not top_words:
                print(f"      → No words found for {cat}")
                continue

            # Print top-10 preview
            preview = ", ".join(
                f"{w}({f})" for w, f in top_words[:10]
            )
            print(f"    {ICON[cat]} Top-10 preview [{cat}]: {preview}")

            for rank, (word, freq) in enumerate(top_words, start=1):
                row = {
                    "model": label,
                    "subject": subject,
                    "trend": cat,
                    "rank": rank,
                    "word": word,
                    "freq": freq,
                    "n_topics_in_category": len(trend_topics[cat]),
                }
                per_subject_rows.append(row)
                all_rows.append(row)

        # Save per-model per-subject CSV
        if per_subject_rows:
            sub_df = pd.DataFrame(per_subject_rows)
            out_path = (
                RESULTS_DIR / "shared" / "trend_model" / model / subject
                / "trend_top_words.csv"
            )
            sub_df.to_csv(out_path, index=False)
            print(f"    💾 Saved → {out_path}")

print("\n" + "="*70)
print("✅ Done")


Model: DTM

  [CS]
    📈 GROWING: 14 topics
    🔒 STABLE: 12 topics
    📉 DECLINING: 24 topics
    📈 Top-10 preview [GROWING]: channel(90), theory(72), logic(70), program(67), scheme(66), language(63), random(59), order(56), quantum(51), power(43)
    🔒 Top-10 preview [STABLE]: channel(108), program(62), theory(60), logic(54), property(47), random(46), probability(45), power(44), set(43), matrix(42)
    📉 Top-10 preview [DECLINING]: channel(187), theory(156), program(149), random(137), logic(128), protocol(102), set(92), probability(92), power(92), game(82)
    💾 Saved → /home/nedo/Kuliah/TA/Program/results/shared/trend_model/dtm/cs/trend_top_words.csv

  [MATH]
    📈 GROWING: 7 topics
    🔒 STABLE: 32 topics
    📉 DECLINING: 11 topics
    📈 Top-10 preview [GROWING]: theory(80), algebra(59), non(56), group(51), manifold(48), space(47), finite(47), invariant(47), field(43), class(41)
    🔒 Top-10 preview [STABLE]: structure(269), algebra(260), theory(260), manifold(243), space(199), op

    🔒 Top-10 preview [STABLE]: channel(95), network(91), word(84), language(84), agent(75), datum(68), algorithm(62), text(62), sentence(56), antenna(55)
    📉 Top-10 preview [DECLINING]: graph(143), algorithm(124), code(85), vertex(84), polynomial(69), edge(53), logic(51), approximation(51), document(48), language(44)
    💾 Saved → /home/nedo/Kuliah/TA/Program/results/shared/trend_model/topicGpt/cs/trend_top_words.csv

  [MATH]
    📈 GROWING: 26 topics
    🔒 STABLE: 32 topics
    📉 DECLINING: 32 topics
    📈 Top-10 preview [GROWING]: graph(110), vertex(96), function(93), algorithm(87), random(79), operator(71), edge(71), stochastic(69), optimization(69), estimate(65)
    🔒 Top-10 preview [STABLE]: group(209), random(140), algebra(113), subgroup(105), representation(98), graph(98), function(94), finite(94), field(86), tree(85)
    📉 Top-10 preview [DECLINING]: algebra(261), group(222), manifold(198), surface(159), quantum(152), module(151), curve(146), cohomology(105), algebras(105), v

    💾 Saved → /home/nedo/Kuliah/TA/Program/results/shared/trend_model/topicGpt/math/trend_top_words.csv

  [PHYSICS]
    📈 GROWING: 24 topics
    🔒 STABLE: 32 topics
    📉 DECLINING: 18 topics
    📈 Top-10 preview [GROWING]: optical(147), photonic(78), detector(76), quantum(72), energy(70), light(64), nonlinear(58), metamaterial(58), photon(57), molecular(51)
    🔒 Top-10 preview [STABLE]: quantum(166), electron(132), beam(113), atom(113), plasma(106), energy(93), optical(92), ion(85), network(85), laser(74)
    📉 Top-10 preview [DECLINING]: atom(105), laser(92), optical(75), energy(58), quantum(53), turbulence(52), ion(48), relativity(45), scale(45), atomic(44)
    💾 Saved → /home/nedo/Kuliah/TA/Program/results/shared/trend_model/topicGpt/physics/trend_top_words.csv

✅ Done


## Save Combined CSV (All Models × Subjects)

In [5]:
combined_df = pd.DataFrame(all_rows)

combined_path = RESULTS_DIR / "shared" / "trend_model" / "trend_top_words_all.csv"
combined_df.to_csv(combined_path, index=False)

print(f"Combined CSV: {combined_path}")
print(f"Total rows  : {len(combined_df)}")
print(f"\nSample (first 15 rows):")
print(combined_df.head(15).to_string(index=False))

Combined CSV: /home/nedo/Kuliah/TA/Program/results/shared/trend_model/trend_top_words_all.csv
Total rows  : 900

Sample (first 15 rows):
model subject   trend  rank       word  freq  n_topics_in_category
  DTM      cs GROWING     1    channel    90                    14
  DTM      cs GROWING     2     theory    72                    14
  DTM      cs GROWING     3      logic    70                    14
  DTM      cs GROWING     4    program    67                    14
  DTM      cs GROWING     5     scheme    66                    14
  DTM      cs GROWING     6   language    63                    14
  DTM      cs GROWING     7     random    59                    14
  DTM      cs GROWING     8      order    56                    14
  DTM      cs GROWING     9    quantum    51                    14
  DTM      cs GROWING    10      power    43                    14
  DTM      cs GROWING    11      error    39                    14
  DTM      cs GROWING    12   property    39               

## Summary Table: Top-20 Words per Model × Subject × Category

In [6]:
for model_label in MODEL_LABELS.values():
    print(f"\n{'='*80}")
    print(f"  {model_label}")
    print(f"{'='*80}")

    m_df = combined_df[combined_df["model"] == model_label]

    for subject in LIST_SUBJECT:
        s_df = m_df[m_df["subject"] == subject]
        if s_df.empty:
            continue

        print(f"\n  [{subject.upper()}]")
        for cat in TREND_CATEGORIES:
            c_df = s_df[s_df["trend"] == cat].sort_values("rank")
            if c_df.empty:
                continue
            words_str = ", ".join(
                f"{r['word']}({r['freq']})" for _, r in c_df.iterrows()
            )
            n_topics = c_df.iloc[0]["n_topics_in_category"]
            print(f"    {ICON[cat]} {cat} ({n_topics} topics):")
            print(f"      {words_str}")


  DTM

  [CS]
    📈 GROWING (14 topics):
      channel(90), theory(72), logic(70), program(67), scheme(66), language(63), random(59), order(56), quantum(51), power(43), error(39), property(39), class(37), set(36), particular(36), protocol(34), optimal(33), complexity(32), information(32), approximation(32)
    🔒 STABLE (12 topics):
      channel(108), program(62), theory(60), logic(54), property(47), random(46), probability(45), power(44), set(43), matrix(42), protocol(42), semantic(41), constraint(41), service(41), complexity(37), quantum(35), order(35), scheme(34), game(32), language(31)
    📉 DECLINING (24 topics):
      channel(187), theory(156), program(149), random(137), logic(128), protocol(102), set(92), probability(92), power(92), game(82), complexity(81), possible(77), scheme(77), tree(73), service(71), datum(68), function(66), order(65), language(62), quantum(61)

  [MATH]
    📈 GROWING (7 topics):
      theory(80), algebra(59), non(56), group(51), manifold(48), space(47), 

## Cross-Model Aggregation: Top-20 Words per Subject × Trend Category

Gabungkan **semua model** untuk subjek yang sama:
- Kumpulkan seluruh kata dari topik-topik GROWING/STABLE/DECLINING di semua model
- Hitung frekuensi gabungan (jumlah kemunculan kata di semua entri year×topic dari semua model)
- Ambil top-20 kata per (subjek, kategori)

**Output:** `results/shared/trend_model/trend_top_words_subject.csv`

In [7]:
subject_rows = []

print("Cross-Model Top-20 Words per Subject x Trend Category")
print("(aggregated across ALL models)\n")

for subject in LIST_SUBJECT:
    print(f"{'='*70}")
    print(f"Subject: {subject.upper()}")
    print(f"{'='*70}")

    # Counter gabungan per kategori untuk subjek ini
    combined_counter = {cat: Counter() for cat in TREND_CATEGORIES}
    n_topics_agg = {cat: 0 for cat in TREND_CATEGORIES}  # total topik lintas model

    for model in MODELS:
        trend_topics = load_trend_topics(model, subject)
        evo_df = load_word_evolution(model, subject)

        if evo_df.empty:
            continue

        for cat in TREND_CATEGORIES:
            topic_ids = trend_topics[cat]
            n_topics_agg[cat] += len(topic_ids)

            if not topic_ids:
                continue

            subset = evo_df[evo_df["topic_id"].isin(topic_ids)]
            for top_words_str in subset["top_words"]:
                words = [w.strip() for w in str(top_words_str).split(",") if w.strip()]
                combined_counter[cat].update(words)

    # Ambil top-20 per kategori dan simpan
    for cat in TREND_CATEGORIES:
        top_words = combined_counter[cat].most_common(TOP_N)
        n_topics = n_topics_agg[cat]

        if not top_words:
            print(f"  {ICON[cat]} {cat}: (no data)")
            continue

        preview = ", ".join(f"{w}({f})" for w, f in top_words[:10])
        print(f"  {ICON[cat]} {cat} ({n_topics} topics across all models):")
        print(f"    Top-10: {preview}")

        for rank, (word, freq) in enumerate(top_words, start=1):
            subject_rows.append({
                "subject": subject,
                "trend": cat,
                "rank": rank,
                "word": word,
                "freq": freq,
                "n_topics_all_models": n_topics,
            })

    print()

# Simpan CSV
subject_df = pd.DataFrame(subject_rows)
subject_out = RESULTS_DIR / "shared" / "trend_model" / "trend_top_words_subject.csv"
subject_df.to_csv(subject_out, index=False)
print(f"💾 Saved → {subject_out}")
print(f"   Total rows: {len(subject_df)} (3 subjects x 3 categories x {TOP_N} words)")

Cross-Model Top-20 Words per Subject x Trend Category
(aggregated across ALL models)

Subject: CS
  📈 GROWING (456 topics across all models):
    Top-10: image(1298), datum(838), dataset(735), network(643), object(507), algorithm(392), language(388), detection(385), visual(373), video(372)
  🔒 STABLE (218 topics across all models):
    Top-10: network(548), algorithm(535), datum(519), channel(479), language(284), agent(259), word(255), document(217), text(194), wireless(190)
  📉 DECLINING (130 topics across all models):
    Top-10: algorithm(555), graph(357), logic(339), protocol(333), program(312), quantum(303), channel(295), language(294), network(277), polynomial(261)

Subject: MATH


  📈 GROWING (184 topics across all models):
    Top-10: algorithm(549), graph(453), network(350), vertex(338), stochastic(317), numerical(303), function(300), optimization(269), control(245), edge(233)
  🔒 STABLE (192 topics across all models):
    Top-10: function(663), group(632), random(564), space(529), algebra(496), operator(419), distribution(404), field(386), manifold(372), graph(361)
  📉 DECLINING (160 topics across all models):
    Top-10: group(910), algebra(888), manifold(672), space(524), surface(438), algebras(410), function(388), module(381), quantum(363), operator(355)

Subject: PHYSICS


  📈 GROWING (237 topics across all models):
    Top-10: optical(694), flow(391), quantum(377), detector(297), energy(275), light(240), fluid(233), photon(231), datum(226), photonic(225)
  🔒 STABLE (230 topics across all models):
    Top-10: electron(605), energy(546), quantum(501), wave(490), atom(467), laser(466), flow(450), beam(446), network(439), plasma(411)
  📉 DECLINING (146 topics across all models):
    Top-10: energy(541), atom(411), electron(408), quantum(367), laser(330), beam(322), theory(316), field(295), wave(263), optical(243)

💾 Saved → /home/nedo/Kuliah/TA/Program/results/shared/trend_model/trend_top_words_subject.csv
   Total rows: 180 (3 subjects x 3 categories x 20 words)


## Summary Table: Cross-Model Top-20 per Subject

In [8]:
for subject in LIST_SUBJECT:
    print(f"\n{'='*80}")
    print(f"  {subject.upper()} — Top-20 Words per Trend Category (All Models Combined)")
    print(f"{'='*80}")

    s_df = subject_df[subject_df["subject"] == subject]

    for cat in TREND_CATEGORIES:
        c_df = s_df[s_df["trend"] == cat].sort_values("rank")
        if c_df.empty:
            continue

        n_topics = int(c_df.iloc[0]["n_topics_all_models"])
        print(f"\n  {ICON[cat]} {cat}  ({n_topics} topics across all models)")
        print(f"  {'Rank':>4}  {'Word':<30}  {'Freq':>6}")
        print(f"  {'-'*4}  {'-'*30}  {'-'*6}")
        for _, r in c_df.iterrows():
            print(f"  {int(r['rank']):>4}  {r['word']:<30}  {int(r['freq']):>6}")


  CS — Top-20 Words per Trend Category (All Models Combined)

  📈 GROWING  (456 topics across all models)
  Rank  Word                              Freq
  ----  ------------------------------  ------
     1  image                             1298
     2  datum                              838
     3  dataset                            735
     4  network                            643
     5  object                             507
     6  algorithm                          392
     7  language                           388
     8  detection                          385
     9  visual                             373
    10  video                              372
    11  training                           371
    12  text                               332
    13  learning                           329
    14  human                              329
    15  recognition                        327
    16  deep                               324
    17  robot                              321
